In [1]:
!pip install gradio

In [2]:
# ==========================================
# HEALTHCARE COST PREDICTION
# RANDOM FOREST + GRID SEARCH + GRADIO
# ==========================================

# Step 1: Import Libraries
import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# ==========================================
# Step 2: Load Dataset
# ==========================================

df = pd.read_csv("healthcare_cost_prediction_100k.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())


# ==========================================
# Step 3: Remove Patient ID
# ==========================================

df = df.drop("patient_id", axis=1)


# ==========================================
# Step 4: Features and Target
# ==========================================

X = df.drop("healthcare_cost", axis=1)
y = df["healthcare_cost"]


# ==========================================
# Step 5: Feature Types
# ==========================================

numeric_features = [
    "age",
    "bmi",
    "children",
    "chronic_conditions",
    "blood_pressure",
    "cholesterol",
    "annual_income"
]

categorical_features = [
    "sex",
    "smoker",
    "region",
    "exercise_level"
]


# ==========================================
# Step 6: Preprocessing
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


# ==========================================
# Step 7: Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ==========================================
# Step 8: Random Forest Model
# ==========================================

model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)


# ==========================================
# Step 9: Pipeline
# ==========================================

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)


# ==========================================
# Step 10: Grid Search
# ==========================================

param_grid = {
    "model__n_estimators": [100, 150],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)


print("\nRunning Grid Search...")

grid_search.fit(X_train, y_train)

print("\nGrid Search Completed!")


# ==========================================
# Step 11: Best Model
# ==========================================

best_model = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest CV R² Score:")
print(grid_search.best_score_)


# ==========================================
# Step 12: Model Evaluation
# ==========================================

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)


print("\n========== MODEL PERFORMANCE ==========")

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)


# ==========================================
# Step 13: Gradio Prediction Function
# ==========================================

def predict_healthcare_cost(
    age,
    sex,
    bmi,
    children,
    smoker,
    region,
    exercise_level,
    chronic_conditions,
    blood_pressure,
    cholesterol,
    annual_income
):

    new_patient = pd.DataFrame({
        "age": [age],
        "sex": [sex],
        "bmi": [bmi],
        "children": [children],
        "smoker": [smoker],
        "region": [region],
        "exercise_level": [exercise_level],
        "chronic_conditions": [chronic_conditions],
        "blood_pressure": [blood_pressure],
        "cholesterol": [cholesterol],
        "annual_income": [annual_income]
    })

    prediction = best_model.predict(new_patient)

    return f"Predicted Healthcare Cost: ₹{prediction[0]:,.2f}"


# ==========================================
# Step 14: Create Gradio Interface
# ==========================================

demo = gr.Interface(

    fn=predict_healthcare_cost,

    inputs=[
        gr.Number(label="Age"),
        
        gr.Dropdown(
            ["male", "female"],
            label="Sex"
        ),

        gr.Number(label="BMI"),

        gr.Number(label="Children"),

        gr.Dropdown(
            ["yes", "no"],
            label="Smoker"
        ),

        gr.Dropdown(
            ["northeast", "northwest", "southeast", "southwest"],
            label="Region"
        ),

        gr.Dropdown(
            ["low", "moderate", "high"],
            label="Exercise Level"
        ),

        gr.Number(label="Chronic Conditions"),

        gr.Number(label="Blood Pressure"),

        gr.Number(label="Cholesterol"),

        gr.Number(label="Annual Income")
    ],

    outputs=gr.Textbox(
        label="Healthcare Cost Prediction"
    ),

    title="🏥 Healthcare Cost Prediction",

    description=(
        "Enter patient details to predict "
        "the estimated healthcare cost using "
        "Random Forest and GridSearchCV."
    )
)


# ==========================================
# Step 15: Launch Gradio
# ==========================================

demo.launch()

Dataset Shape: (100000, 13)

First 5 Rows:
   patient_id  age     sex   bmi  children smoker     region exercise_level  \
0           1   56  female  34.5         2     no  northwest           high   
1           2   46  female  23.7         0     no  northeast           high   
2           3   32  female  32.7         2     no  southwest       moderate   
3           4   60    male  26.2         0    yes  northwest       moderate   
4           5   25    male  33.4         4     no  northeast           high   

   chronic_conditions  blood_pressure  cholesterol  annual_income  \
0                   1           106.9        237.0        42100.0   
1                   1           132.2        217.7        55446.0   
2                   0           142.7        198.1        34517.0   
3                   0           132.2        175.6        64779.0   
4                   1           122.7        167.1       102490.0   

   healthcare_cost  
0         12387.37  
1          5974.34  
2   